In [ ]:
cd(joinpath(@__DIR__, ".."))   # run from the repository root

# Hodge stars with one value per edge: Laplacian, spectrum and Darcy fluxes

**Question.** We want a discrete Hodge star $H$ that is *consistent*,
$H\,\alpha(p)=\beta(p)$ for every polynomial 1-form $p\in[P_k]^2$ (circulations on the
Delaunay edges in, fluxes through the Voronoi edges out), and *symmetric*, so that
$L=d_0^\top H d_0$ has a real, positive spectrum. With one value per edge this is
impossible beyond a mesh-dependent rate; the proof is in the two notes:

* **[F]** *Symmetric Hodge stars with one unknown per edge: an accuracy floor*
  (`hodge_symmetry_floor.pdf`): Facts F1–F7, Lemmas 1–7, Theorems 1–3.
* **[N]** *Flux consistency versus symmetry for discrete Hodge stars on Delaunay–Voronoi
  meshes* (`report/hodge_symmetry_consistency.pdf`): full proofs and experiments.

This notebook shows the **consequences** with the juliaDEC code, for five stars:

| star | juliaDEC | symmetric | exact (flux) on |
|---|---|---|---|
| diagonal $H_0=\mathrm{diag}(\omega_e)$, $\omega_e=\lvert\tilde e\rvert/\lvert e\rvert$ | `Diagonal(ω(m))` | yes | $[P_0]^2$ |
| lift $[P_1]^2$ | `hodge_star_lift_matrix(m; only_linear=true)` | no | $[P_1]^2$ (13-edge stencil) |
| lift $[P_2]^2$ | `hodge_star_lift_matrix(m; only_linear=false)` | no | $[P_2]^2$ (19-edge stencil) |
| sandwich $[P_1]^2$ | `hodge_star_sandwich_matrix(m; only_linear=true)` | yes | none (energy exact on $[P_1]^2$) |
| sandwich $[P_2]^2$ | `hodge_star_sandwich_matrix(m; only_linear=false)` | yes | none (energy exact on $[P_2]^2$) |

The lift is your construction: on the stencil of edge $e$, fit the coefficients of the
basis $(x-x_m)^\alpha$ to the circulations by least squares ($\Pi_e=A_e^{+}$), then
multiply by the vector $\int_{\star e}p_i$. The sandwich is
$\sum_e \Pi_e^\top M_e\Pi_e$ plus a stabilisation, with $M_e$ the $L^2$ mass matrix of
the basis on the diamond $D_e=\mathrm{conv}(e\cup\tilde e)$.

| § | computed here | statement it illustrates |
|---|---|---|
| 1 | meshes, offsets $\delta_e$, circumradius jumps | [F] Facts F1, F2, Lemma 7 |
| 2 | the five stars: symmetry, coefficient size, flux (Hodge) error | [F] eqs. (3)–(4), Lemma 3; [N] Prop. 8.5 |
| 3 | Laplacian consistency: truncation and Poisson solution | [N] Props. 12.3, 12.6 |
| 4 | eigenvalues of $L$ | [F] Lemma 1, Sec. 11; [N] Props. 9.1, 9.4, Cor. 9.5 |
| 5 | Darcy (mixed) problem: why the sandwich gets the fluxes wrong | [N] Props. 8.5, 8.6; [F] Sec. 12 |
| 6 | the floor itself: $s_h$, Lemma 1, Lemma 2 | [F] Lemmas 1–2, Theorem 1, hypothesis H1 |
| 7 | summary | |

Running everything takes a few minutes (building the stars and the dense eigensolves).

## 0. Setup and notation

The notation of [F, Section 2] and where it lives in the code:

| [F] | meaning | code |
|---|---|---|
| $J$ | rotation by $+90^\circ$; $\star u = Ju$ | `_ROT90`, `star_field(u)` |
| $\alpha_e(w)=\int_e w\cdot t_e\,ds$ | circulation on the Delaunay edge (primal reduction) | `reduce_primal_edges(m, w).values` |
| $\beta_e(w)=\int_{\tilde e}(Jw)\cdot n_e\,ds=\int_{\tilde e}w\cdot t_e\,ds$ | flux through the Voronoi edge (dual reduction of $\star w$) | `reduce_dual_edges(m, star_field(w)).values` |
| $\varepsilon(w)=H\alpha(w)-\beta(w)$ | flux error = Hiptmair's Hodge error, eq. (3) | `ε(H, m, w)` below |
| $K=\tfrac12(H-H^\top)$, $\Lambda(H)=\max_e\sum_m\lvert H_{em}\rvert$ | skew part, coefficient size | `skw(H)`, `Λ(H)` |
| $\omega_e=\lvert\tilde e\rvert/\lvert e\rvert$, $\lVert x\rVert_W$, $\lVert y\rVert_{W^{-1}}$ | DEC weights and norms | `ω(m)`, `normW`, `normWinv` |
| $a_h(u,v)=\alpha(v)^\top\beta(u)$, $s_h=a_h(u,v)-a_h(v,u)$ | exact pairing and its skew part, eq. (5) | `s_h(m, u, v)` |
| $d_0$, $D_1$ | gradient (edges × generators), curl (triangles × edges) | `d0_matrix_primal`, `d1_matrix_primal` |

In [ ]:
using VoronoiMeshes, TensorsLite, TensorsLiteGeometry, SmallCollections, LinearAlgebra, NCDatasets
using StaticArrays, SparseArrays, QuadGK, Printf
using CairoMakie
# TensorsLite pirates similar(::Broadcasted, ::Type{<:AbstractTensor}); this also catches Union{} (same fix as whitney_basis.ipynb)
Base.similar(bc::Base.Broadcast.Broadcasted, ::Type{Union{}}) = Array{Union{}}(undef, length.(axes(bc)))
include("src/juliaDEC.jl")
include("src/hodge_stars.jl")
set_theme!(Theme(fontsize = 13, Axis = (titlefont = :regular, titlesize = 13), Legend = (framevisible = false, labelsize = 11)))

In [ ]:
# ---------------------------------------------------------------- the objects of [F, Section 2]
α(m, w) = reduce_primal_edges(m, w).values                    # circulations α_e(w) = ∫_e w·t_e
β(m, w) = reduce_dual_edges(m, star_field(w)).values          # fluxes β_e(w) = ∫_ẽ w·t_e
ω(m) = edge_lengths(m, false) ./ edge_lengths(m, true)        # ω_e = |ẽ|/|e|  (H0 = diag ω)
normW(m, x) = sqrt(sum(ω(m) .* x .^ 2))                       # ‖x‖_W      (circulation-type)
normWinv(m, y) = sqrt(sum(y .^ 2 ./ ω(m)))                    # ‖y‖_{W^-1} (flux-type)
ε(H, m, w) = H * α(m, w) - β(m, w)                            # Hodge error, [F] eq. (3)
skw(H) = (H - H') / 2                                         # K
Λ(H) = maximum(sum(abs.(H), dims = 2))                        # coefficient size
s_h(m, u, v) = dot(α(m, v), β(m, u)) - dot(α(m, u), β(m, v))  # [F] eq. (5)
meanh(m) = sum(edge_lengths(m, true)) / m.edges.n
d0m(m) = sparse(Float64.(d0_matrix_primal(m)))
D1m(m) = sparse(Float64.(d1_matrix_primal(m)))
rates(h, e) = [log(e[i] / e[i+1]) / log(h[i] / h[i+1]) for i in 1:length(h)-1]
fmt(x) = @sprintf("%.6e", x)
nothing

## 1. Meshes and the offset $\delta_e$

Two families on the unit torus, both well-centred: **regular** (triangular lattice) and
**SCVT** (Lloyd-converged, with pentagon–heptagon defects).

The geometric fact behind everything ([F] Fact F1): the Voronoi edge $\tilde e$ lies on
the perpendicular bisector of $e$, so the circumcentres are $c_i=m_e+s_in_e$ with
$s_1<0<s_2$. Hence $\lvert\tilde e\rvert=s_2-s_1$ and the **dual midpoint** is
$\tilde m_e=m_e+\delta_e n_e$ with $\delta_e=\tfrac12(s_1+s_2)$. The circulation reads a
field around $m_e$, the flux reads it around $\tilde m_e$: the two reading points differ
by $\delta_e$. Fact F2: $\lvert\tilde e\rvert\,\delta_e=\tfrac12(R_2^2-R_1^2)$, the jump
of the squared circumradius across $e$; on a lattice the two triangles are congruent, so
$\delta_e=0$ (Lemma 7).

In [ ]:
FAMILIES = ["regular" => ["mesh1_regular", "mesh2_regular", "mesh3_regular", "mesh4_regular", "mesh5_regular"],
            "SCVT"    => ["mesh1","mesh2", "mesh3", "mesh4", "mesh5"]]
ALLMESHES = vcat(last.(FAMILIES)...)
MESH = Dict(mn => VoronoiMesh("meshes/$mn.nc") for mn in ALLMESHES)

println(rpad("mesh", 15), rpad("cells", 7), rpad("edges", 7), rpad("h", 9), rpad("5/7-gons", 10),
        rpad("mis-oriented dual", 19), "badly-centred")
for mn in ALLMESHES
    m = MESH[mn]
    n5 = count(c -> length(m.cells.edges[c]) == 5, 1:m.cells.n)
    n7 = count(c -> length(m.cells.edges[c]) == 7, 1:m.cells.n)
    _, _, nbad = check_diamond_partition(m)
    println(rpad(mn, 15), rpad(m.cells.n, 7), rpad(m.edges.n, 7), rpad(round(meanh(m), digits = 4), 9),
            rpad("$n5/$n7", 10), rpad(check_dual_orientation(m), 19), nbad)
end

In [ ]:
"δ_e (with n_e = J t_e), |ẽ| and ½(R₂² − R₁²), computed from the geometry ([F] Facts F1, F2)."
function offset_data(m)
    nE = m.edges.n
    δ, Ld, jump = zeros(nE), zeros(nE), zeros(nE)
    for e in 1:nE
        xa, xb = edge_endpoints(m, e, true)          # Delaunay edge, t_e from x_a to x_b
        c1, c2 = edge_endpoints(m, e, false)         # Voronoi edge, c2 - c1 along n_e
        t = (xb - xa) / norm(xb - xa); n = [-t[2], t[1]]
        δ[e] = dot((c1 + c2) / 2 - (xa + xb) / 2, n)
        Ld[e] = norm(c2 - c1)
        jump[e] = (norm(c2 - xa)^2 - norm(c1 - xa)^2) / 2   # R_i = |c_i - x_a|
    end
    return δ, Ld, jump
end

println(rpad("mesh", 15), rpad("max |δ|", 14), "max | |ẽ|δ − ½(R₂²−R₁²)")
for mn in ALLMESHES
    δ, Ld, jump = offset_data(MESH[mn])
    r = abs.(δ) 
    println(rpad(mn, 15), rpad(fmt(maximum(r)), 14),
            fmt(maximum(abs.(Ld .* δ .- jump))))
end

In [ ]:
let f = Figure(size = (1000, 470))
    defect_pts(m, k) = [Point2f(_pt(m.cells.position[c])...) for c in 1:m.cells.n if length(m.cells.edges[c]) == k]
    for (i, mn) in enumerate(["mesh3_regular", "mesh3"])
        m = MESH[mn]; δ, Ld, _ = offset_data(m); r = δ ./ Ld
        ax = Axis(f[1, i], aspect = DataAspect(), title = "$mn: Delaunay edges coloured by δₑ/|ẽ|")
        hidedecorations!(ax); hidespines!(ax)
        segs = Point2f[]; cols = Float64[]
        for e in 1:m.edges.n
            a, b = edge_endpoints(m, e, true)
            push!(segs, Point2f(a...), Point2f(b...)); push!(cols, r[e], r[e])
        end
        linesegments!(ax, segs, color = cols, colormap = :balance, colorrange = (-0.22, 0.22), linewidth = 2)
        scatter!(ax, defect_pts(m, 5), marker = :diamond, color = :black, markersize = 10)
        scatter!(ax, defect_pts(m, 7), marker = :star5, color = :black, markersize = 11)
        xlims!(ax, 0, 1); ylims!(ax, 0, 1)
    end
    Colorbar(f[1, 3], colormap = :balance, limits = (-0.22, 0.22))
    f
end

On the lattice $\delta_e$ vanishes to the precision of the stored coordinates; on the SCVT
mesh the large offsets sit at the pentagon (◆) / heptagon (★) pairs, and their maximum does
not decrease under refinement. Fact F2 holds to round-off on every mesh.

## 2. The five stars: symmetry and flux consistency

Consistency is measured by the **Hodge error** $\varepsilon(w)=H\alpha(w)-\beta(w)$ in the
norm $\lVert\cdot\rVert_{W^{-1}}$ ([F] eq. (3); by Satoh–Yaguchi this is an $L^2$ norm).
[F] Lemma 3: a local star that is row-wise exact on $[P_k]^2$ with bounded $\Lambda(H)$ has
$\lVert\varepsilon(w)\rVert_{W^{-1}}=O(h^{k+1})$. Two probes: a **constant** field (every
star exact on $[P_0]^2$ must reproduce it to round-off; it is periodic, so no seam
effect) and a smooth periodic field.

In [ ]:
NAMES = ["diagonal", "lift P1", "lift P2", "sandwich P1", "sandwich P2"]
COLORS = Dict("diagonal" => :gray40, "lift P1" => :darkorange, "lift P2" => :teal,
              "sandwich P1" => :mediumpurple, "sandwich P2" => :crimson)
build_stars(m) = Dict("diagonal"    => sparse(Diagonal(ω(m))),
                      "lift P1"     => hodge_star_lift_matrix(m; only_linear = true),
                      "lift P2"     => hodge_star_lift_matrix(m; only_linear = false),
                      "sandwich P1" => hodge_star_sandwich_matrix(m; only_linear = true),
                      "sandwich P2" => hodge_star_sandwich_matrix(m; only_linear = false))
@time STARS = Dict(mn => build_stars(MESH[mn]) for mn in ALLMESHES);

In [ ]:
for mn in ("mesh4_regular", "mesh4")
    println("--- ", mn)
    println(rpad("star", 14), rpad("nnz/row", 9), rpad("‖K‖/‖H‖", 12), "Λ(H)")
    for nm in NAMES
        H = STARS[mn][nm]
        println(rpad(nm, 14), rpad(round(nnz(H) / size(H, 1), digits = 1), 9),
                rpad(fmt(norm(skw(H)) / norm(H)), 12), round(Λ(H), digits = 2))
    end
end

In [ ]:
A_CONST(x) = [1.0, 0.7]
U(x) = [sin(2pi * x[2]), cos(2pi * x[1])]                       # smooth periodic test field
relHodge(m, H, w) = normWinv(m, ε(H, m, w)) / normWinv(m, β(m, w))

FLUX = Dict()     # (mesh, star) => (constant, smooth) relative Hodge error
for mn in ALLMESHES, nm in NAMES
    m = MESH[mn]; H = STARS[mn][nm]
    FLUX[(mn, nm)] = (relHodge(m, H, A_CONST), relHodge(m, H, U))
end
for (fam, mns) in FAMILIES
    println("=== ", fam, ":  relative Hodge error ‖ε(w)‖_{W⁻¹}/‖β(w)‖_{W⁻¹}   [constant a | smooth U]")
    println(rpad("mesh", 15), join([rpad(nm, 24) for nm in NAMES]))
    for mn in mns
        println(rpad(mn, 15), join([rpad(fmt(FLUX[(mn, nm)][1]) * " | " * fmt(FLUX[(mn, nm)][2]), 24) for nm in NAMES]))
    end
    hs = [meanh(MESH[mn]) for mn in mns]
    println(rpad("rates (smooth)", 15), join([rpad(join(round.(rates(hs, [FLUX[(mn, nm)][2] for mn in mns]), digits = 1), ", "), 24) for nm in NAMES]))
end

In [ ]:
let f = Figure(size = (1000, 400))
    for (j, (fam, mns)) in enumerate(FAMILIES)
        hs = [meanh(MESH[mn]) for mn in mns]
        ax = Axis(f[1, j], xscale = log10, yscale = log10, xreversed = true, xlabel = "h",
                  ylabel = j == 1 ? "‖ε(U)‖_{W⁻¹} / ‖β(U)‖_{W⁻¹}" : "", title = "$fam: flux (Hodge) error, smooth field")
        for nm in NAMES
            e = [FLUX[(mn, nm)][2] for mn in mns]
            scatterlines!(ax, hs, e, color = COLORS[nm], label = nm, linewidth = 2)
        end
        lines!(ax, hs, 0.5 * (hs ./ hs[1]) .^ 2, color = :black, linestyle = :dot, label = "h²")
        lines!(ax, hs, 0.05 * (hs ./ hs[1]) .^ 3, color = :black, linestyle = :dash, label = "h³")
        j == 2 && axislegend(ax, position = :lb)
    end
    f
end

**Reading.** The lifts are exact on constants to round-off and converge on smooth fields
at least at the rate of their basis: $h^2$ for $[P_1]^2$; for $[P_2]^2$, $h^4$ on the lattice
and about $h^{3.5}$ on SCVT meshes (Lemma 3 guarantees $h^3$). They are genuine discrete
Hodge stars. The diagonal star is exact on constants and converges at $h^2$ on the lattice
and about $h^{1.6}$ on SCVT meshes, the rate allowed by the offsets. The **sandwich** fails
the constant test: its flux error on a constant field is 0.18–0.20 on the lattice and
0.10–0.22 on SCVT meshes, and it does not go to zero. This is
[N] Proposition 8.5: on a constant field $a$ the sandwich returns
$(B\alpha(a))_m=a\cdot\sum_{e:\,m\in S_e}\int_{D_e}\phi_{e,m}\,dx$, a sum of fitted fields
over the diamonds, while the exact flux is $a\cdot\lvert\tilde m\rvert t_m$; nothing in
the construction ties the two. The sandwich was built to reproduce *energies*
($z^\top Bz=\int\lvert w\rvert^2$ on $[P_k]^2$), which constrains only the projection of its
fluxes onto the reduced polynomials ([N] Prop. 8.4).

## 3. Laplacian consistency

$L=d_0^\top H d_0$ acts on values at the generators. Two different questions:

1. **Pointwise truncation**:
   $\tau_c=\frac1{\lvert V_c\rvert}(d_0^\top Hd_0\,p)_c+\frac1{\lvert V_c\rvert}\int_{V_c}\Delta p$.
   Because $d_0$ and $d_0^\top$ are exact (Stokes), [N] Prop. 12.3 gives the exact identity
   $$\tau_c=\frac1{\lvert V_c\rvert}\big(d_0^\top\varepsilon(\nabla p)\big)_c :$$
   the truncation is the discrete divergence of the star's flux error, divided by an area.
2. **Convergence of the Poisson solution** $Lp_h=b$, $b_c=\int_{V_c}f$: by [N] Prop. 12.6,
   $z^\top Hz=-z^\top\varepsilon(\nabla p)$ for $z=d_0(p_h-p)$, so only the part of the flux
   error seen by **gradients** matters.

Test function $p=\sin2\pi x\sin2\pi y+0.3\cos2\pi(x-2y)$.

In [ ]:
pex(x)  = sin(2pi * x[1]) * sin(2pi * x[2]) + 0.3cos(2pi * (x[1] - 2x[2]))
gpex(x) = [2pi * cos(2pi * x[1]) * sin(2pi * x[2]) - 0.6pi * sin(2pi * (x[1] - 2x[2])),
           2pi * sin(2pi * x[1]) * cos(2pi * x[2]) + 1.2pi * sin(2pi * (x[1] - 2x[2]))]
lapp(x) = -8pi^2 * sin(2pi * x[1]) * sin(2pi * x[2]) - 6pi^2 * cos(2pi * (x[1] - 2x[2]))

"τ (max norm, relative to max|Δp|) and the defect of the identity τ = d₀ᵀε(∇p)/|V|."
function truncation(m, H)
    d0 = d0m(m)
    p = [pex(_pt(m.cells.position[c])) for c in 1:m.cells.n]
    avg = discretize_scalar_field_cells(m, lapp).values ./ m.cells.area
    τ = (d0' * (H * (d0 * p))) ./ m.cells.area .+ avg
    τε = (d0' * ε(H, m, gpex)) ./ m.cells.area
    sc = maximum(abs, avg)
    return maximum(abs, τ) / sc, maximum(abs, τ .- τε) / sc
end

"Relative error of the Poisson solution at the generators (area-weighted, mean removed)."
function poisson_error(m, H)
    ph = solve_poisson_primal(m, H, x -> -lapp(x))
    pe = [pex(_pt(m.cells.position[c])) for c in 1:m.cells.n]; pe .-= sum(pe) / length(pe)
    A = m.cells.area
    return sqrt(sum(A .* (ph .- pe) .^ 2) / sum(A .* pe .^ 2))
end

LAP = Dict()
for mn in ALLMESHES, nm in NAMES
    m = MESH[mn]; H = STARS[mn][nm]
    τ, idd = truncation(m, H)
    LAP[(mn, nm)] = (τ = τ, identity = idd, poisson = poisson_error(m, H))
end
println("identity τ = d₀ᵀε(∇p)/|V| ([N] Prop. 12.3): largest defect over all meshes and stars = ",
        fmt(maximum(LAP[k].identity for k in keys(LAP))), "  (quadrature level)\n")
for (fam, mns) in FAMILIES
    hs = [meanh(MESH[mn]) for mn in mns]
    println("=== ", fam)
    println(rpad("", 22), join([rpad(nm, 14) for nm in NAMES]))
    for mn in mns
        println(rpad("max|τ| " * mn, 22), join([rpad(fmt(LAP[(mn, nm)].τ), 14) for nm in NAMES]))
    end
    for mn in mns
        println(rpad("Poisson " * mn, 22), join([rpad(fmt(LAP[(mn, nm)].poisson), 14) for nm in NAMES]))
    end
    println(rpad("Poisson rates", 22), join([rpad(join(round.(rates(hs, [LAP[(mn, nm)].poisson for mn in mns]), digits = 1), ","), 14) for nm in NAMES]))
end

In [ ]:
let f = Figure(size = (1000, 720))
    for (j, (fam, mns)) in enumerate(FAMILIES)
        hs = [meanh(MESH[mn]) for mn in mns]
        ax1 = Axis(f[1, j], xscale = log10, yscale = log10, xreversed = true, xlabel = "h",
                   ylabel = j == 1 ? "max_c |τ_c| / max|Δp|" : "", title = "$fam: pointwise truncation of L")
        ax2 = Axis(f[2, j], xscale = log10, yscale = log10, xreversed = true, xlabel = "h",
                   ylabel = j == 1 ? "‖p_h − p‖ / ‖p‖" : "", title = "$fam: Poisson solution error")
        for nm in NAMES
            scatterlines!(ax1, hs, [LAP[(mn, nm)].τ for mn in mns], color = COLORS[nm], label = nm, linewidth = 2)
            scatterlines!(ax2, hs, [LAP[(mn, nm)].poisson for mn in mns], color = COLORS[nm], label = nm, linewidth = 2)
        end
        lines!(ax2, hs, 0.05 * (hs ./ hs[1]) .^ 2, color = :black, linestyle = :dot, label = "h²")
        lines!(ax2, hs, 0.05 * (hs ./ hs[1]) .^ 4, color = :black, linestyle = :dash, label = "h⁴")
        j == 2 && Legend(f[1:2, 3], ax2, framevisible = false)
    end
    f
end

**Reading.** *All five Laplacians converge*: the Poisson solution error decreases under
refinement for every star, on both families: at second order for the diagonal star, the
$[P_1]^2$ lift and both sandwiches, at fourth order for the $[P_2]^2$ lift. (On the coarsest
SCVT mesh the $[P_1]^2$ lift gives a useless solution: its Laplacian is indefinite, §4, and
nearly singular there.) This is the sense in which every star "works" for the Laplacian,
and why the sandwich looked fine: by [N] Prop. 12.6 the solution error is controlled by
the part of $\varepsilon(\nabla p)$ seen by **gradients**, in a weak (dual) norm, and that part is
small even when the flux error itself is $O(1)$.

The pointwise truncation tells a finer story. A star exact on $[P_1]^2$ (both lifts) gives
a pointwise-consistent Laplacian on both families ([N] Cor. 12.4). The diagonal star is
pointwise inconsistent on SCVT meshes ($\max\lvert\tau\rvert$ stalls near $2\times10^{-2}$: Peixoto 2016;
$\tau_c$ is the isotropy defect of the cell, of size $\max\delta_e/h$) but its solution still
converges at second order ("supraconvergence"). The sandwiches are pointwise consistent
on the lattice, but not on SCVT meshes: there $\max\lvert\tau\rvert$ stalls near 0.3 for $[P_1]^2$ and
*grows* for $[P_2]^2$ (0.14 → 0.64). Their $O(1)$ flux error, divided by a cell area, is visible
pointwise; the solution converges anyway for the reason above.

## 4. Eigenvalues of the Laplacian

Generalised problem $Lp=\lambda S_0p$, $S_0=\mathrm{diag}(\lvert V_c\rvert)$, solved densely
on meshes 2–4. The exact eigenvalues on the unit torus are $4\pi^2(j^2+k^2)$.

Why the stars behave differently ([N] Prop. 9.1, with $H=S+K$):
$$\operatorname{Re}\lambda=\frac{z^*Sz}{p^*S_0p},\qquad \operatorname{Im}\lambda=\frac{-i\,z^*Kz}{p^*S_0p},\qquad z=d_0p .$$
* a symmetric star ($K=0$: diagonal, sandwich) has a **real** spectrum, positive if $S$ is
  positive on gradients;
* a non-symmetric star (the lifts) can have **complex** eigenvalues, and by [F] Lemma 1 a
  consistent star *must* be non-symmetric: its $K$ has to carry $\tfrac12 s_h$
  ([F] Sec. 11, point 4);
* the **sign** of $\operatorname{Re}\lambda$ depends on $\operatorname{sym}H$ on gradients: for
  the $[P_1]^2$ lift it is indefinite, which gives negative eigenvalues growing like
  $h^{-2}$ in number $\propto N$ (proved on lattices, [N] Prop. 9.4 and Cor. 9.5).

In [ ]:
function spectrum(m, H)
    d0 = d0m(m); s = 1 ./ sqrt.(m.cells.area)
    A = Matrix(Diagonal(s) * (d0' * H * d0) * Diagonal(s))
    norm(A - A') <= 1e-12 * norm(A) && return complex.(eigvals(Symmetric((A + A') / 2)))
    return eigvals(A)
end
function exact_eigs(m, n)
    Kmax = 30
    v = sort([4pi^2 * ((j / m.x_period)^2 + (k / m.y_period)^2) for j in -Kmax:Kmax for k in -Kmax:Kmax])
    return v[1:n]
end
"Smallest eigenvalue of sym(H) restricted to gradients (dense; only for meshes up to ~1000 cells)."
symgrad_min(m, H) = m.cells.n > 1100 ? NaN :
    (Q = Matrix(qr(Matrix(d0m(m))).Q)[:, 1:m.cells.n-1];          # orthonormal basis of range(d₀)
     eigmin(Symmetric(Q' * Matrix((H + H') / 2) * Q)))

EIG_MESHES = ALLMESHES                                  # dense eigensolves up to 4096 cells (a few minutes)
@time EIG = Dict((mn, nm) => spectrum(MESH[mn], STARS[mn][nm]) for mn in EIG_MESHES for nm in NAMES);

In [ ]:
NLOW = 20          # the modes with |k|² ≤ 5 (the band of the test function p of §3)
nonneg_sorted(λ) = sort(filter(x -> real(x) > -1e-8, λ), by = real)
println(rpad("mesh", 15), rpad("star", 13), rpad("#Re<0", 7), rpad("min Re λ", 11), rpad("max|Im λ|", 11),
        rpad("max|Im|/|λ|", 13), rpad("low-mode err", 14), "min eig sym(H) on grads")
SPEC = Dict()
for mn in EIG_MESHES, nm in NAMES
    m = MESH[mn]; λ = EIG[(mn, nm)]
    λs = nonneg_sorted(λ)                                    # spurious negative eigenvalues left out
    ex = exact_eigs(m, NLOW + 1)
    low = maximum(abs.(real.(λs[2:NLOW+1]) .- ex[2:NLOW+1]) ./ ex[2:NLOW+1])
    nneg = count(x -> real(x) < -1e-8, λ)
    SPEC[(mn, nm)] = (nneg = nneg, minre = minimum(real, λ), maxim = maximum(abs ∘ imag, λ), low = low)
    sg = symgrad_min(m, STARS[mn][nm])
    println(rpad(mn, 15), rpad(nm, 13), rpad(nneg, 7), rpad(fmt(minimum(real, λ)), 11), rpad(fmt(maximum(abs ∘ imag, λ)), 11),
            rpad(fmt(maximum(abs.(imag.(λ)) ./ max.(abs.(λ), 1e-12))), 13), rpad(fmt(low), 14),
            isnan(sg) ? "—" : round(sg, digits = 3))
end

In [ ]:
let f = Figure(size = (1100, 420)), mn = "mesh4"
    m = MESH[mn]
    ax1 = Axis(f[1, 1], xlabel = "Re λ", ylabel = "Im λ", title = "SCVT mesh4: eigenvalues in the complex plane")
    for nm in NAMES
        λ = EIG[(mn, nm)]
        scatter!(ax1, real.(λ), imag.(λ), color = (COLORS[nm], 0.6), markersize = nm == "diagonal" ? 9 : 5, label = nm)
    end
    axislegend(ax1, position = :lb)
    ax2 = Axis(f[1, 2], xlabel = "index i", ylabel = "Re λᵢ", title = "lowest 150 modes with Re λ ≥ 0, vs exact")
    ex = exact_eigs(m, 150)
    for nm in NAMES
        λs = nonneg_sorted(EIG[(mn, nm)])
        lines!(ax2, 1:150, real.(λs[1:150]), color = COLORS[nm], linewidth = 2, label = nm)
    end
    scatter!(ax2, 1:150, ex, color = :black, markersize = 4, label = "exact 4π²(j²+k²)")
    axislegend(ax2, position = :lt)
    ax3 = Axis(f[1, 3], xscale = log10, yscale = log10, xreversed = true, xlabel = "h",
               ylabel = "max relative error, modes 2–$(NLOW+1)", title = "low modes (|k|² ≤ 5), SCVT")
    mns = ["mesh2", "mesh3", "mesh4", "mesh5"]; hs = [meanh(MESH[x]) for x in mns]
    for nm in NAMES
        scatterlines!(ax3, hs, [SPEC[(x, nm)].low for x in mns], color = COLORS[nm], linewidth = 2, label = nm)
    end
    f
end

**Energy consistency versus flux consistency, seen in the eigenvalues.** Take an exact
eigenfunction, $p=\cos2\pi x\cos2\pi y$ ($\lambda=8\pi^2$), sampled at the generators, and two
Rayleigh quotients:
$$R_{\rm lumped}=\frac{(d_0p)^\top H(d_0p)}{\sum_c\lvert V_c\rvert p_c^2},\qquad
R_{\rm cons}=\frac{(d_0p)^\top H(d_0p)}{\sum_c p_c\int_{V_c}p\,dx}.$$
* By [N] Lemma 12.2, $(d_0p)^\top H(d_0p)=\lambda\sum_cp_c\int_{V_c}p+(d_0p)^\top\varepsilon(\nabla p)$. So
  $R_{\rm cons}-\lambda$ is exactly the **flux error** of the star, paired with $d_0p$.
* On these meshes $\sum_c\lvert V_c\rvert p_c^2$ equals $\int p^2$ almost exactly (to $10^{-15}$ on the lattice and
  $3\times10^{-6}$ on SCVT mesh5, checked separately), and $\lambda\int p^2=\int\lvert\nabla p\rvert^2$. So
  $R_{\rm lumped}-\lambda$ measures the **energy error** $(d_0p)^\top H(d_0p)-\int\lvert\nabla p\rvert^2$.

The generalised eigenproblem $Lp=\lambda S_0p$ uses the lumped mass, so its accuracy follows
$R_{\rm lumped}$, i.e. the energy consistency of the star.

In [ ]:
pmode(x) = cos(2pi * x[1]) * cos(2pi * x[2]); λmode = 8pi^2
for (fam, mns) in FAMILIES
    hs = [meanh(MESH[mn]) for mn in mns]
    println("=== ", fam, ":  |R/λ − 1| for p = cos2πx cos2πy     [lumped mass | consistent mass]")
    println(rpad("mesh", 15), join([rpad(nm, 24) for nm in NAMES]))
    R = Dict()
    for mn in mns
        m = MESH[mn]; p = [pmode(_pt(m.cells.position[c])) for c in 1:m.cells.n]
        z = d0m(m) * p; ip = discretize_scalar_field_cells(m, pmode).values
        for nm in NAMES
            num = dot(z, STARS[mn][nm] * z)
            R[(mn, nm)] = (abs(num / sum(m.cells.area .* p .^ 2) / λmode - 1), abs(num / dot(p, ip) / λmode - 1))
        end
        println(rpad(mn, 15), join([rpad(fmt(R[(mn, nm)][1]) * " | " * fmt(R[(mn, nm)][2]), 24) for nm in NAMES]))
    end
    println(rpad("rates (cons.)", 15), join([rpad(join(round.(rates(hs, [R[(mn, nm)][2] for mn in mns]), digits = 1), ", "), 24) for nm in NAMES]))
end

**Reading.**
* **diagonal, sandwich $[P_1]^2$, sandwich $[P_2]^2$** (symmetric): real spectrum, no
  negative eigenvalue, $\operatorname{sym}H$ positive on gradients, and the low modes converge to
  $4\pi^2(j^2+k^2)$. This is what [F] Lemma 1 allows: with $K=0$ the eigenvalues are real.
* **lift $[P_2]^2$**: real parts positive ($\operatorname{sym}H$ positive on gradients) and low modes
  converging, but on SCVT meshes the spectrum is **complex**, with $\lvert\operatorname{Im}\lambda\rvert$ up to
  15–28% of $\lvert\lambda\rvert$ at the grid scale (meshes 4–5). The imaginary parts are forced: a consistent star must
  have $K\neq0$ ([F] Lemma 1, §6 below), and $\operatorname{Im}\lambda$ is $K$ seen by $z=d_0p$ ([N] Prop. 9.1).
  On the lattice its spectrum is real, because plane waves are exact eigenvectors there ([N] Prop. 9.4).
* **lift $[P_1]^2$**: $\operatorname{sym}H$ is indefinite on gradients (last column, $\approx-0.17$ on every
  mesh), and the spectrum has negative real parts: 10 → 42 → 168 → 680 on the lattice (22% of $N$)
  with the most negative one multiplied by exactly 4 per level ($h^{-2}$, [N] Cor. 9.5), and the
  same behaviour on SCVT meshes (14 → 57 → 226 → 896). Even its non-negative part is polluted:
  spurious grid-scale eigenvalues near zero mix with the smooth ones, so its low-mode error
  does not converge. Its Rayleigh quotients on a smooth mode do converge (second order): the
  trouble is the grid-scale modes, not the smooth ones.
* **Accuracy of the low modes, and the two consistencies.** The Rayleigh table separates them.
  The $[P_2]^2$ **lift** has a tiny flux error ($R_{\rm cons}$: fourth order) but an $O(h^2)$ energy error
  ($R_{\rm lumped}$: second order). The $[P_2]^2$ **sandwich** is the mirror image: its energy is nearly
  exact ($R_{\rm lumped}$: $10^{-5}$ on mesh5, about fourth order) but its flux error is $O(h^2)$ when paired
  with a gradient. The diagonal star and the $[P_1]^2$ stars are second order in both. Because the
  eigenproblem uses the lumped mass, the **sandwich $[P_2]^2$ gives the most accurate low
  eigenvalues** (about third order in the table), even though it is the worst star for fluxes.
  This is [N] Prop. 8.4 in numbers: the sandwich was built for energies, the lift for fluxes,
  and with one value per edge a symmetric star cannot be both (§6).

So the eigenvalue picture is the mirror of §2: the stars with the right *fluxes* (lifts)
have the wrong *spectrum*, and the stars with the right spectrum (symmetric) cannot all
have the right fluxes. §5 shows what that costs for the sandwich.

## 5. The Darcy problem: the sandwich gets the fluxes wrong

Mixed problem with $u=-J\nabla p$ (so $\star u=\nabla p$ and $\operatorname{curl}u=-\Delta p$),
unknowns $f$ (primal 1-cochain, circulations) and $p_h$ (values at the circumcentres):
$$Hf-D_1^\top p_h=0,\qquad D_1f=\varphi,\qquad \varphi_T=\int_T(-\Delta p)\,dx .$$
The first equation says "flux through each dual edge = pressure difference along it", so
here $H$ is used as a **map** from circulations to fluxes, exactly the property the
sandwich lacks.

[N] Prop. 8.6 (mixed error identity): if $z^\top\operatorname{sym}(H)z\ge c\lVert z\rVert_W^2$ on
$\ker D_1$, then
$$\lVert f-\alpha(u)\rVert_W\le c^{-1}\lVert\varepsilon(u)\rVert_*,\qquad
\lVert\varepsilon\rVert_*=\sup_{D_1z=0}\frac{z^\top\varepsilon}{\lVert z\rVert_W},$$
i.e. only the part of the flux error seen by **discretely divergence-free circulations**
reaches the computed flux; the rest goes into the pressure.

In [ ]:
uex(x) = (g = gpex(x); [g[2], -g[1]])          # u = −J∇p :  ⋆u = ∇p,  curl u = −Δp

function darcy(m, H)
    D1 = D1m(m); nE, nT = m.edges.n, m.vertices.n
    φ = discretize_scalar_field_triangles_ho(m, x -> -lapp(x)).values; φ .-= sum(φ) / nT
    Kmat = [sparse(H) -sparse(D1'); D1 spzeros(nT, nT)]
    keep = setdiff(1:nE+nT, [nE + 1])            # pin one pressure (constants)
    sol = zeros(nE + nT)
    sol[keep] = Kmat[keep, keep] \ [zeros(nE); φ][keep]
    f, p = sol[1:nE], sol[nE+1:end]
    fex = α(m, uex)
    A = m.vertices.area
    pT = [pex(_pt(m.vertices.position[t])) for t in 1:nT]
    pp = p .- sum(A .* p) / sum(A); pe = pT .- sum(A .* pT) / sum(A)
    return (flux = normW(m, f - fex) / normW(m, fex),
            pressure = sqrt(sum(A .* (pp .- pe) .^ 2) / sum(A .* pe .^ 2)), f = f, fex = fex)
end

@time DARCY = Dict((mn, nm) => darcy(MESH[mn], STARS[mn][nm]) for mn in ALLMESHES for nm in NAMES);
for (fam, mns) in FAMILIES
    hs = [meanh(MESH[mn]) for mn in mns]
    println("=== ", fam, ":  Darcy  [flux error | pressure error]")
    println(rpad("mesh", 15), join([rpad(nm, 24) for nm in NAMES]))
    for mn in mns
        println(rpad(mn, 15), join([rpad(fmt(DARCY[(mn, nm)].flux) * " | " * fmt(DARCY[(mn, nm)].pressure), 24) for nm in NAMES]))
    end
    println(rpad("flux rates", 15), join([rpad(join(round.(rates(hs, [DARCY[(mn, nm)].flux for mn in mns]), digits = 1), ", "), 24) for nm in NAMES]))
end

In [ ]:
"‖ε(u)‖_W⁻¹ (total), ‖ε(u)‖_* (part seen by ker D₁), coercivity c of sym H on ker D₁, all relative."
function mixed_analysis(m, H, Z, ZWZ)
    e = ε(H, m, uex); ref = normWinv(m, β(m, uex))
    g = Z' * e
    estar = sqrt(dot(g, ZWZ \ g))
    c = minimum(eigvals(Symmetric(Z' * Matrix((H + H') / 2) * Z), ZWZ))
    return (total = normWinv(m, e) / ref, divfree = estar / ref, c = c)
end

MIXED_MESHES = ["mesh2_regular", "mesh3_regular", "mesh4_regular", "mesh2", "mesh3", "mesh4"]   # dense SVD of D₁
MIXED = Dict()
for mn in MIXED_MESHES
    m = MESH[mn]
    Z = nullspace(Matrix(D1m(m)))                 # orthonormal basis of ker D₁ (full SVD)
    ZWZ = Symmetric(Z' * (ω(m) .* Z))
    for nm in NAMES
        MIXED[(mn, nm)] = mixed_analysis(m, STARS[mn][nm], Z, ZWZ)
    end
end
println(rpad("mesh", 15), rpad("star", 13), rpad("‖ε‖_W⁻¹", 11), rpad("‖ε‖_*", 11), rpad("c", 8),
        rpad("bound c⁻¹‖ε‖_*", 16), "measured flux error")
for mn in MIXED_MESHES, nm in NAMES
    r = MIXED[(mn, nm)]; d = DARCY[(mn, nm)]
    # the bound in flux units: c⁻¹‖ε‖_* relative to ‖β(u)‖_{W⁻¹}, converted to ‖α(u)‖_W
    m = MESH[mn]; conv = normWinv(m, β(m, uex)) / normW(m, d.fex)
    println(rpad(mn, 15), rpad(nm, 13), rpad(fmt(r.total), 11), rpad(fmt(r.divfree), 11), rpad(round(r.c, digits = 2), 8),
            rpad(r.c > 0 ? fmt(r.divfree / r.c * conv) : "c ≤ 0", 16), fmt(d.flux))
end

In [ ]:
let f = Figure(size = (1100, 420))
    for (j, (fam, mns)) in enumerate(FAMILIES)
        hs = [meanh(MESH[mn]) for mn in mns]
        ax = Axis(f[1, j], xscale = log10, yscale = log10, xreversed = true, xlabel = "h",
                  ylabel = j == 1 ? "‖f − α(u)‖_W / ‖α(u)‖_W" : "", title = "$fam: Darcy flux error")
        for nm in NAMES
            scatterlines!(ax, hs, [DARCY[(mn, nm)].flux for mn in mns], color = COLORS[nm], linewidth = 2, label = nm)
        end
        lines!(ax, hs, 0.05 * (hs ./ hs[1]) .^ 2, color = :black, linestyle = :dot, label = "h²")
        j == 1 && axislegend(ax, position = :lb)
    end
    mns = ["mesh2", "mesh3", "mesh4"]; hs = [meanh(MESH[x]) for x in mns]
    ax = Axis(f[1, 3], xscale = log10, yscale = log10, xreversed = true, xlabel = "h",
              title = "SCVT: sandwich P2 flux error, split")
    scatterlines!(ax, hs, [MIXED[(x, "sandwich P2")].total for x in mns], color = :crimson, linewidth = 2, label = "‖ε‖ total")
    scatterlines!(ax, hs, [MIXED[(x, "sandwich P2")].divfree for x in mns], color = :crimson, linestyle = :dash, linewidth = 2, label = "‖ε‖_* (seen by ker D₁)")
    scatterlines!(ax, hs, [MIXED[(x, "lift P2")].divfree for x in mns], color = :teal, linestyle = :dash, linewidth = 2, label = "lift P2: ‖ε‖_*")
    axislegend(ax, position = :lb)
    f
end

**Reading.**
* **lift $[P_2]^2$**: fluxes converge at high order on both families ($2.4\times10^{-5}$ on SCVT mesh5).
* **diagonal**: second order on the lattice, about $h^{1.6}$ on SCVT meshes ($4.5\times10^{-4}$ on mesh5).
  For it the bound of Prop. 8.6 is an equality ($c=1$ in the $W$ norm).
* **sandwiches on SCVT meshes**: the flux error decreases only slowly and irregularly
  (local rates 0.3–1.0) and is $8.5\times10^{-3}$ ($[P_2]^2$) and $1.2\times10^{-2}$ ($[P_1]^2$) on mesh5:
  about 20 times worse than the diagonal star and 350 times worse than the lift. The
  reason is in the table: the part of the Hodge error seen by divergence-free circulations,
  $\lVert\varepsilon\rVert_*$, decays slowly ($4.6\times10^{-2}\to1.7\times10^{-2}$ for $[P_2]^2$), and Prop. 8.6 says the
  computed flux cannot be better; the measured error sits just below the bound.
* **sandwiches on the lattice**: there $\lVert\varepsilon\rVert_*$ of the $[P_2]^2$ sandwich happens to converge
  fast, so its flux converges too, but the $O(1)$ defect goes into the **pressure**, which
  converges at about first order ($7.4\times10^{-3}$ on mesh5, against $7.9\times10^{-6}$ for the lift).
* **lift $[P_1]^2$**: $\operatorname{sym}H$ is indefinite on $\ker D_1$ ($c<0$), so Prop. 8.6 gives nothing; its
  flux converges only at about first order on SCVT meshes and fails on the coarsest one.

This is the precise sense of "the sandwich is not capable of obtaining the right fluxes":
it is not a symmetric approximation of a Hodge star, it is a symmetric bilinear form that
is right on energies and unconstrained on fluxes ([N] Prop. 8.4, [F] Sec. 12: its
reconstruction does not commute with $D_1$). The failure is the construction's, not the
mesh's: [F] Theorem 1 would allow a symmetric star with second-order fluxes on SCVT
meshes (e.g. the symmetric $[P_1]^2$-exact star of [F] Theorem 3).

## 6. The floor: $s_h$, Lemma 1 and Lemma 2 of [F]

Everything above comes from one mesh number,
$s_h(u,v)=\alpha(v)^\top\beta(u)-\alpha(u)^\top\beta(v)$, which does not involve $H$.

* **Lemma 1** (pure linear algebra, any $H$):
  $2\alpha(v)^\top K\alpha(u)=s_h(u,v)+\alpha(v)^\top\varepsilon(u)-\alpha(u)^\top\varepsilon(v)$.
  For a consistent star ($\varepsilon\to0$) the skew part must reproduce $s_h$: the lift's
  asymmetry is forced.
* **Lemma 2** (the floor, symmetric $H$):
  $\max(\lVert\varepsilon(u)\rVert_{W^{-1}},\lVert\varepsilon(v)\rVert_{W^{-1}})\ge
  \dfrac{\lvert s_h(u,v)\rvert}{\sqrt{2\lvert\Omega\rvert}(\lVert u\rVert_\infty+\lVert v\rVert_\infty)}$.
* **Theorem 1** and hypothesis **H1**: if $\lvert s_h\rvert\ge c\,h^q$, no symmetric star has a
  flux error better than $O(h^q)$. On SCVT meshes $q\approx2$ is *measured* here, not proved.

In [ ]:
V(x) = [cos(2pi * (x[1] + x[2])), sin(2pi * (x[1] - 2x[2]))]
supnorm(w) = maximum(norm(w([x, y])) for x in range(0, 1, length = 301) for y in range(0, 1, length = 301))
NU, NV = supnorm(U), supnorm(V)

for mn in ["pert2", "pert3", "pert4", "pert5"]       # a third family, used only for s_h
    MESH[mn] = VoronoiMesh("meshes/$mn.nc")
end
SH = Dict(mn => s_h(MESH[mn], U, V) for mn in keys(MESH))
for (fam, mns) in vcat(FAMILIES, ["perturbed" => ["pert2", "pert3", "pert4", "pert5"]])
    hs = [meanh(MESH[mn]) for mn in mns]; s = [abs(SH[mn]) for mn in mns]
    println(rpad(fam, 11), "|s_h(U,V)| = ", join(fmt.(s), "  "), "    local rates ", join(round.(rates(hs, s), digits = 1), ", "))
end

In [ ]:
println("Lemma 1 on every mesh and star:  | 2α(v)ᵀKα(u) − [s_h + α(v)ᵀε(u) − α(u)ᵀε(v)] |")
println("  relative to the sum of the absolute products it is assembled from (the round-off scale;")
println("  U and V are nearly L²-orthogonal, so the pairings themselves are small sums of O(1) products)")
worst = 0.0
for mn in ALLMESHES, nm in NAMES
    m = MESH[mn]; H = STARS[mn][nm]
    au, av, bu, bv = α(m, U), α(m, V), β(m, U), β(m, V)
    lhs = 2 * dot(av, skw(H) * au)
    rhs = SH[mn] + dot(av, H * au - bu) - dot(au, H * av - bv)
    absH = abs.(H)
    scale = sum(abs.(av .* bu)) + sum(abs.(au .* bv)) + dot(abs.(av), absH * abs.(au)) + dot(abs.(au), absH * abs.(av))
    global worst = max(worst, abs(lhs - rhs) / scale)
end
println("  relative defect ≤ ", fmt(worst), "   (round-off: the identity is exact)\n")

println("The lift's skew part carries s_h (SCVT):   2α(V)ᵀKα(U)   vs   s_h(U,V)")
for mn in ["mesh2", "mesh3", "mesh4", "mesh5"]
    m = MESH[mn]
    for nm in ("lift P1", "lift P2")
        K2 = 2 * dot(α(m, V), skw(STARS[mn][nm]) * α(m, U))
        println("  ", rpad(mn, 7), rpad(nm, 9), "2α(V)ᵀKα(U) = ", rpad(fmt(K2), 11), " s_h = ", rpad(fmt(SH[mn]), 11), " ratio ", round(K2 / SH[mn], digits = 3))
    end
end

In [ ]:
Ω = 1.0                                             # |Ω| of the unit torus
println("Lemma 2 (symmetric stars):  floor = |s_h| / (√(2|Ω|)(‖U‖∞+‖V‖∞))   vs   max(‖ε(U)‖, ‖ε(V)‖) in W⁻¹")
println(rpad("mesh", 15), rpad("floor", 11), join([rpad(nm, 14) for nm in ("diagonal", "sandwich P1", "sandwich P2")]))
for mn in ALLMESHES
    m = MESH[mn]
    floor_ = abs(SH[mn]) / (sqrt(2Ω) * (NU + NV))
    errs = [max(normWinv(m, ε(STARS[mn][nm], m, U)), normWinv(m, ε(STARS[mn][nm], m, V))) for nm in ("diagonal", "sandwich P1", "sandwich P2")]
    println(rpad(mn, 15), rpad(fmt(floor_), 11), join([rpad(fmt(x), 14) for x in errs]))
end

**Reading.** $s_h$ is negligible on the lattice (offsets vanish), decays roughly like $h^2$
on SCVT meshes and faster, irregularly, on the perturbed lattice ([F] Sec. 10 table).
Lemma 1 holds to round-off for every star, symmetric or not. For the $[P_2]^2$ lift, whose
flux errors ($\sim h^{3.5}$) are much smaller than $s_h$ ($\sim h^2$), $2\alpha(V)^\top K\alpha(U)$ approaches
$s_h(U,V)$ (ratio 0.99 and 1.02 on meshes 4 and 5): the asymmetry of the lift is not a
defect of the fit but the mesh skew showing through, as Lemma 1 says it must. For the
$[P_1]^2$ lift the error terms are $O(h^2)$, the same order as $s_h$, so its $K$ carries mostly
them (ratio ≈ 10); the identity still holds exactly.
The symmetric stars sit above the Lemma 2 floor, as they must; the floor itself is small
on these meshes, which is why it caps the *rate* (Theorem 1) rather than the size of the
error on one mesh.

## 7. Summary

| star | flux (Hodge) error, §2 | Laplacian / Poisson, §3 | spectrum, §4 | Darcy flux, §5 | where in the notes |
|---|---|---|---|---|---|
| diagonal | exact on constants; $h^2$ (lattice), $h^{1.6}$ (SCVT) | converges ($h^2$); pointwise inconsistent on SCVT | real, positive | $h^2$ / $h^{1.6}$ | [F] Cor. 3; [N] §12 (Peixoto) |
| lift $[P_1]^2$ | $O(h^2)$ | converges ($h^2$); pointwise consistent | **negative** real parts, number $\propto N$, size $\propto h^{-2}$; polluted low end | about first order on SCVT ($c<0$) | [N] Prop. 9.4, Cor. 9.5 |
| lift $[P_2]^2$ | $h^4$ / $h^{3.5}$ | converges at $h^4$ | **complex** on SCVT (grid scale), $\operatorname{Re}>0$ | high order | [F] Lemma 1 (forced $K$); [N] Prop. 9.1 |
| sandwich $[P_1]^2$, $[P_2]^2$ | **$O(1)$**, even on constants | converges ($h^2$); pointwise inconsistent on SCVT | real, positive; $[P_2]^2$ has the most accurate low modes (energy-consistent) | **slow and irregular on SCVT** (rates 0.3–1); on the lattice the pressure pays | [N] Props. 8.4–8.6; [F] Sec. 12 |

**The answer to the question.** With one value per edge, a symmetric star cannot be
flux-consistent beyond the decay rate of $s_h$ ([F] Lemma 2, Theorem 1); demanding exactness on
$[P_2]^2$ with symmetry forces unbounded coefficients on SCVT meshes ([F] Theorem 2,
Cor. 2). The lift chooses consistency and pays in the spectrum; the sandwich chooses
symmetry and, because it was built from energies rather than fluxes, pays even more than
the floor requires: its flux error is $O(1)$. A symmetric star exact on $[P_1]^2$ is the
rate-optimal compromise on SCVT meshes ([F] Theorem 3, under H1–H2); beyond that, the
unknowns have to change ([F] Sec. 12).